In [ ]:
"""
NOTEBOOK: DEMAND FORECASTING
=============================
Purpose: Predict which crops will be in high demand
Output: Demand scores and recommendations
"""

# 📈 Demand Forecasting Notebook

**Objective:** Forecast crop demand based on historical patterns

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

class DemandForecaster:
    """
    PRODUCTION-READY DEMAND FORECASTING
    Exports to: backend/app/ml/demand_forecaster.py
    """
    
    def __init__(self):
        self.model = None
        self.scaler = StandardScaler()
        self.feature_columns = None
    
    def prepare_features(self, df):
        """Prepare features for demand prediction"""
        
        # Aggregate by crop and month
        demand_data = df.groupby(['crop_type', 'month', 'year']).agg({
            'transaction_id': 'count',
            'total_amount': 'sum',
            'quantity_kg': 'sum'
        }).reset_index()
        
        # Create features
        features = pd.DataFrame()
        features['crop_encoded'] = pd.Categorical(demand_data['crop_type']).codes
        features['month_sin'] = np.sin(2 * np.pi * demand_data['month'] / 12)
        features['month_cos'] = np.cos(2 * np.pi * demand_data['month'] / 12)
        features['prev_month_demand'] = demand_data.groupby('crop_type')['transaction_id'].shift(1)
        features['prev_month_demand'] = features['prev_month_demand'].fillna(0)
        
        # Target: current month demand
        target = demand_data['transaction_id']
        
        self.feature_columns = features.columns.tolist()
        
        return features, target
    
    def train(self, df):
        """Train demand forecasting model"""
        print("📊 Training demand forecasting model...")
        
        X, y = self.prepare_features(df)
        
        # Scale features
        X_scaled = self.scaler.fit_transform(X)
        
        # Train Random Forest
        self.model = RandomForestRegressor(
            n_estimators=100,
            max_depth=10,
            random_state=42,
            n_jobs=-1
        )
        self.model.fit(X_scaled, y)
        
        # Calculate feature importance
        importance = pd.DataFrame({
            'feature': self.feature_columns,
            'importance': self.model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        print(f"✅ Model trained. R²: {self.model.score(X_scaled, y):.3f}")
        print(f"\n📊 Feature Importance:")
        for _, row in importance.head(5).iterrows():
            print(f"   {row['feature']}: {row['importance']:.3f}")
        
        return self.model
    
    def predict_demand(self, crop_type, month):
        """Predict demand for specific crop and month"""
        if self.model is None:
            raise ValueError("Model not trained yet")
        
        # Prepare features for prediction
        features = pd.DataFrame({
            'crop_encoded': [pd.Categorical([crop_type]).codes[0]],
            'month_sin': [np.sin(2 * np.pi * month / 12)],
            'month_cos': [np.cos(2 * np.pi * month / 12)],
            'prev_month_demand': [0]  # Will use average
        })
        
        features_scaled = self.scaler.transform(features)
        demand_score = self.model.predict(features_scaled)[0]
        
        # Normalize to 0-100
        demand_score_normalized = min(100, (demand_score / 100) * 100)
        
        return {
            'crop': crop_type,
            'month': month,
            'demand_score': round(demand_score_normalized, 1),
            'demand_level': self._get_demand_level(demand_score_normalized)
        }
    
    def _get_demand_level(self, score):
        """Convert score to demand level"""
        if score > 70:
            return 'HIGH'
        elif score > 40:
            return 'MEDIUM'
        else:
            return 'LOW'
    
    def save_model(self, version="v1"):
        """Save model to production"""
        import joblib
        import os
        
        os.makedirs("../../backend/ml_weights", exist_ok=True)
        
        model_path = f"../../backend/ml_weights/demand_model_{version}.pkl"
        joblib.dump(self.model, model_path)
        
        scaler_path = f"../../backend/ml_weights/demand_scaler_{version}.pkl"
        joblib.dump(self.scaler, scaler_path)
        
        print(f"💾 Demand model saved to: {model_path}")
        return model_path

print("\n✅ Demand Forecaster loaded!")